In [1]:
from pathlib import Path
import pandas as pd, numpy as np, matplotlib.pyplot as plt
pd.set_option("display.max_columns",100)
def locate_csv():
    for p in [Path("Water Quality.csv"),Path.home()/"Desktop/Water Quality.csv",Path.home()/"Downloads/Water Quality.csv",Path.home()/"Documents/Water Quality.csv"]:
        if p.exists(): return p
    try:
        import tkinter as tk
        from tkinter import filedialog
        root=tk.Tk(); root.withdraw()
        f=filedialog.askopenfilename(title="Select Water Quality.csv",filetypes=[("CSV files","*.csv")]); root.destroy()
        if f: return Path(f)
    except Exception: pass
    raise FileNotFoundError("Put Water Quality.csv beside the notebook or update the path.")
CSV_PATH=locate_csv(); df=pd.read_csv(CSV_PATH,low_memory=False)
def find_col(names):
    m={str(c).strip().lower():c for c in df.columns}
    return next((m[n.lower()] for n in names if n.lower() in m),None)
DATE_COL=find_col(["ActivityStartDate","Date","SampleDate","MonitoringDate","timestamp"])
VALUE_COL=find_col(["ResultValue","Value","Measurement"])
PARAM_COL=find_col(["CharacteristicName","Parameter","Analyte"])
SITE_COL=find_col(["MonitoringLocationID","SiteID","LocationID","StationID"])
UNIT_COL=find_col(["ResultUnit","Unit","Units"])
WQI_COL=find_col(["WQI","WaterQualityIndex","Water Quality Index"])
if DATE_COL: df[DATE_COL]=pd.to_datetime(df[DATE_COL],errors="coerce")
if VALUE_COL: df[VALUE_COL]=pd.to_numeric(df[VALUE_COL],errors="coerce")
print("File:",CSV_PATH,"Shape:",df.shape)
print(dict(DATE_COL=DATE_COL,VALUE_COL=VALUE_COL,PARAM_COL=PARAM_COL,SITE_COL=SITE_COL,UNIT_COL=UNIT_COL,WQI_COL=WQI_COL))
display(df.head())

File: Water Quality.csv Shape: (229916, 40)
{'DATE_COL': 'ActivityStartDate', 'VALUE_COL': 'ResultValue', 'PARAM_COL': 'CharacteristicName', 'SITE_COL': 'MonitoringLocationID', 'UNIT_COL': 'ResultUnit', 'WQI_COL': None}


,DatasetName,MonitoringLocationID,MonitoringLocationName,MonitoringLocationLatitude,MonitoringLocationLongitude,MonitoringLocationHorizontalCoordinateReferenceSystem,MonitoringLocationHorizontalAccuracyMeasure,MonitoringLocationHorizontalAccuracyUnit,MonitoringLocationVerticalMeasure,MonitoringLocationVerticalUnit,MonitoringLocationType,ActivityType,ActivityMediaName,ActivityStartDate,ActivityStartTime,ActivityEndDate,ActivityEndTime,ActivityDepthHeightMeasure,ActivityDepthHeightUnit,SampleCollectionEquipmentName,CharacteristicName,MethodSpeciation,ResultSampleFraction,ResultValue,ResultUnit,ResultValueType,ResultDetectionCondition,ResultDetectionQuantitationLimitMeasure,ResultDetectionQuantitationLimitUnit,ResultDetectionQuantitationLimitType,ResultStatusID,ResultComment,ResultAnalyticalMethodID,ResultAnalyticalMethodContext,ResultAnalyticalMethodName,AnalysisStartDate,AnalysisStartTime,AnalysisStartTimeZone,LaboratoryName,LaboratorySampleID
0,Maritime Coastal Basin Long-term Water Quality...,NS01ED0005,MERSEY RIVER BELOW MILL FALLS SOUTH-WEST OF MA...,44.4367,-65.2228,NAD83,NaN,NaN,NaN,NaN,River/Stream,Sample-Routine,Surface Water,2000-01-03,12:00:00,NaN,NaN,NaN,NaN,NaN,Aluminum,NaN,Extractable,254.70,ug/L,Actual,NaN,NaN,ug/L,NaN,Preliminary,NaN,110688,ENV/CANADA,NaN,NaN,NaN,NaN,ECCCLabs,NaN
1,Maritime Coastal Basin Long-term Water Quality...,NS01ED0005,MERSEY RIVER BELOW MILL FALLS SOUTH-WEST OF MA...,44.4367,-65.2228,NAD83,NaN,NaN,NaN,NaN,River/Stream,Sample-Routine,Surface Water,2000-01-03,12:00:00,NaN,NaN,NaN,NaN,NaN,Apparent color,NaN,NaN,133.00,TCU,Actual,NaN,NaN,TCU,NaN,Preliminary,NaN,2011,ENV/CANADA,NaN,NaN,NaN,NaN,ECCCLabs,NaN
2,Maritime Coastal Basin Long-term Water Quality...,NS01ED0005,MERSEY RIVER BELOW MILL FALLS SOUTH-WEST OF MA...,44.4367,-65.2228,NAD83,NaN,NaN,NaN,NaN,River/Stream,Sample-Routine,Surface Water,2000-01-03,12:00:00,NaN,NaN,NaN,NaN,NaN,Calcium,NaN,Dissolved,1.10,mg/L,Actual,NaN,NaN,mg/L,NaN,Preliminary,NaN,20110,ENV/CANADA,NaN,NaN,NaN,NaN,ECCCLabs,NaN
3,Maritime Coastal Basin Long-term Water Quality...,NS01ED0005,MERSEY RIVER BELOW MILL FALLS SOUTH-WEST OF MA...,44.4367,-65.2228,NAD83,NaN,NaN,NaN,NaN,River/Stream,Sample-Routine,Surface Water,2000-01-03,12:00:00,NaN,NaN,NaN,NaN,NaN,Chloride,NaN,Dissolved,6.13,mg/L,Actual,NaN,NaN,mg/L,NaN,Preliminary,NaN,17209,ENV/CANADA,NaN,NaN,NaN,NaN,ECCCLabs,NaN
4,Maritime Coastal Basin Long-term Water Quality...,NS01ED0005,MERSEY RIVER BELOW MILL FALLS SOUTH-WEST OF MA...,44.4367,-65.2228,NAD83,NaN,NaN,NaN,NaN,River/Stream,Sample-Routine,Surface Water,2000-01-03,12:00:00,NaN,NaN,NaN,NaN,NaN,Gran acid neutralizing capacity,NaN,NaN,-0.24,mg/L,Actual,NaN,NaN,mg/L,NaN,Preliminary,NaN,10110,ENV/CANADA,NaN,NaN,NaN,NaN,ECCCLabs,NaN


In [3]:
if WQI_COL is None:
    print("No observed WQI target column was found. Model training skipped; add validated WQI labels to train a regression model.")
else:
    from sklearn.model_selection import train_test_split
    from sklearn.pipeline import Pipeline
    from sklearn.impute import SimpleImputer
    from sklearn.preprocessing import StandardScaler
    from sklearn.linear_model import LinearRegression
    from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score
    import statsmodels.api as sm
    from statsmodels.stats.outliers_influence import variance_inflation_factor

    # For long-format readings, pivot parameters to columns, then join observed WQI by site/date.
    if all([DATE_COL,VALUE_COL,PARAM_COL]):
        src=df.dropna(subset=[DATE_COL,VALUE_COL]).copy(); src["__date"]=src[DATE_COL].dt.floor("D")
        keys=[c for c in [SITE_COL,"__date"] if c is not None]
        wide=src.pivot_table(index=keys,columns=PARAM_COL,values=VALUE_COL,aggfunc="median").reset_index()
        tgt=df.dropna(subset=[WQI_COL]).copy(); tgt["__date"]=tgt[DATE_COL].dt.floor("D")
        target=tgt.groupby(keys,dropna=False)[WQI_COL].median().reset_index()
        data=wide.merge(target,on=keys,how="inner")
    else: data=df.copy()
    drop=[WQI_COL]+[c for c in [DATE_COL,"__date",SITE_COL,"MonitoringLocationName","MonitoringLocationID"] if c and c in data.columns]
    X=data.drop(columns=drop,errors="ignore").select_dtypes(include=[np.number]).replace([np.inf,-np.inf],np.nan)
    y=pd.to_numeric(data[WQI_COL],errors="coerce")
    valid=y.notna() & X.notna().any(axis=1); X=X.loc[valid]; y=y.loc[valid]
    if len(X)<30: raise ValueError(f"Only {len(X)} usable labelled rows; more labelled data are needed.")
    # Chronological split where possible; otherwise reproducible random split.
    if DATE_COL and DATE_COL in data.columns:
        dates=pd.to_datetime(data.loc[X.index,DATE_COL],errors="coerce")
        order=dates.sort_values().index; cut=int(.8*len(order))
        X_train,X_test=X.loc[order[:cut]],X.loc[order[cut:]]
        y_train,y_test=y.loc[order[:cut]],y.loc[order[cut:]]
    else: X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.2,random_state=42)
    model=Pipeline([("imputer",SimpleImputer(strategy="median")),("scale",StandardScaler()),("regression",LinearRegression())])
    model.fit(X_train,y_train); pred=model.predict(X_test)
    def scores(y_true,y_pred):
        return {"MAE":mean_absolute_error(y_true,y_pred),"RMSE":mean_squared_error(y_true,y_pred)**.5,"R2":r2_score(y_true,y_pred)}
    display(pd.DataFrame([scores(y_test,pred)],index=["Multiple Linear Regression"]))
    print("Features:",list(X.columns)," Train:",len(X_train)," Test:",len(X_test))

No observed WQI target column was found. Model training skipped; add validated WQI labels to train a regression model.


In [5]:
if WQI_COL is None or "y_test" not in globals() or "pred" not in globals():
	print("Residual diagnostics skipped: no WQI target was found, so the model was not trained. Add validated WQI labels and rerun the model cell first.")
else:
	resid=y_test.to_numpy()-pred
	plt.figure(figsize=(6,5)); plt.scatter(pred,resid,alpha=.7); plt.axhline(0,ls="--")
	plt.xlabel("Fitted WQI"); plt.ylabel("Residual"); plt.title("Residuals vs fitted"); plt.grid(alpha=.25); plt.show()
	plt.figure(figsize=(6,5)); plt.scatter(y_test,pred,alpha=.7)
	lims=[min(float(y_test.min()),float(np.min(pred))),max(float(y_test.max()),float(np.max(pred)))]
	plt.plot(lims,lims,ls="--"); plt.xlabel("Actual WQI"); plt.ylabel("Predicted WQI"); plt.title("Actual vs predicted"); plt.grid(alpha=.25); plt.show()
	bp=sm.stats.diagnostic.het_breuschpagan(resid,sm.add_constant(pred))
	print("Breusch-Pagan LM:",round(bp[0],4),"p-value:",round(bp[1],5))
	print("A low p-value may indicate non-constant residual variance; interpret alongside residual plots.")

Residual diagnostics skipped: no WQI target was found, so the model was not trained. Add validated WQI labels and rerun the model cell first.
